# Beating the Casino: Blackjack and Q-learning

A reinforcement-learning agent that learns to play Blackjack from scratch with
tabular **Q-learning**, across six rule variants of growing realism:
the stock game, *double down*, *card counting* (6-deck shoe) and *splitting* pairs.

The reusable pieces live in two modules:

* `blackjack_agent.py` — the environment-agnostic Q-learning agent.
* `blackjack_envs.py` — the custom Gymnasium environments and counting schemes.

Run the full experiment from the command line with `python train.py`.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import gymnasium as gym
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from blackjack_rl.agent import BlackjackAgent, epsilon_greedy, epsilon_greedy_masked
from blackjack_rl.envs import (
    BlackjackDoubleEnv, BlackjackCountEnv, BlackjackSplitEnv,
    COUNT_WEIGHTS_FRACTIONAL, COUNT_WEIGHTS_HI_LO,
)

sns.set_style('whitegrid')
sns.set_palette('colorblind')
SEED = 42

## Experiment parameters

We train each agent for `COUNT_GAMES` episodes, evaluating a fully greedy
policy every `STEP` games to trace the learning curve.

In [ ]:
COUNT_GAMES = 3_500_000
STEP = 35_000
EVAL_GAMES = 20_000
HIT, STICK = 1, 0  # Gymnasium Blackjack-v1: 1 = hit, 0 = stick

## Q-learning update

For each transition $(s, a, r, s')$ the agent moves its estimate toward the
temporal-difference target:

$$Q(s,a) \leftarrow Q(s,a) + \alpha\,[\,r + \gamma \max_{a'} Q(s',a') - Q(s,a)\,]$$

At a **terminal** transition there is no future return, so the target collapses
to just $r$ (the agent handles this internally — see `blackjack_agent.py`).

## Training / evaluation helpers

Exploration `epsilon` **decays** linearly from 0.9 to 0.05 over training:
explore early, exploit late.

In [ ]:
def play_greedy(env, agent):
    state, _ = env.reset()
    terminated = truncated = False
    reward = 0.0
    while not (terminated or truncated):
        action = agent.policy_fn(agent.q, state, 0.0, env)
        state, reward, terminated, truncated, _ = env.step(action)
    return reward


def fit_agent(agent, env, alpha=0.001, gamma=0.95, eps_start=0.9, eps_end=0.05):
    rewards = []
    n_iters = COUNT_GAMES // STEP
    for it in tqdm(range(n_iters)):
        frac = it / max(1, n_iters - 1)
        epsilon = eps_start + (eps_end - eps_start) * frac
        agent.train(STEP, epsilon, alpha, gamma)
        rewards.append(np.mean([play_greedy(env, agent) for _ in range(EVAL_GAMES)]))
    return rewards

## Baseline: fixed-threshold strategy

A non-learning reference that mimics the dealer — keep hitting until the hand
reaches 17, then stand.

In [ ]:
def threshold_strategy_reward(env, threshold=17):
    obs, _ = env.reset()
    player = obs[0]
    terminated = truncated = False
    while player < threshold and not (terminated or truncated):
        obs, reward, terminated, truncated, _ = env.step(HIT)
        player = obs[0]
    if not (terminated or truncated):
        obs, reward, terminated, truncated, _ = env.step(STICK)
    return reward


env = gym.make('Blackjack-v1', natural=True)
env.reset(seed=SEED)
baseline = np.mean([threshold_strategy_reward(env) for _ in range(EVAL_GAMES)])
baseline_rewards = [baseline] * (COUNT_GAMES // STEP)
baseline

## 1. Stock environment (hit / stick)

In [ ]:
env = gym.make('Blackjack-v1', natural=True)
env.reset(seed=SEED)
agent = BlackjackAgent(env, env.action_space.n, epsilon_greedy, seed=SEED)
rewards_default = fit_agent(agent, env)

## 2. Double down

In [ ]:
env = BlackjackDoubleEnv(natural=True)
env.reset(seed=SEED)
agent = BlackjackAgent(env, env.action_space.n, epsilon_greedy, seed=SEED)
rewards_double = fit_agent(agent, env)

## 3. Card counting (fractional weights)

A 6-deck shoe dealt without replacement; the running count is part of the state.

In [ ]:
env = BlackjackCountEnv(natural=True)  # fractional weights by default
env.reset(seed=SEED)
agent = BlackjackAgent(env, env.action_space.n, epsilon_greedy, seed=SEED)
rewards_count = fit_agent(agent, env)

## 4. Card counting (Hi-Lo / plus-minus)

In [ ]:
env = BlackjackCountEnv(natural=True, weights=COUNT_WEIGHTS_HI_LO)
env.reset(seed=SEED)
agent = BlackjackAgent(env, env.action_space.n, epsilon_greedy, seed=SEED)
rewards_count_hi_lo = fit_agent(agent, env)

## 5. Counting + splitting (fractional weights)

Adds the *split* action; the masked policy keeps the agent to legal moves.

In [ ]:
env = BlackjackSplitEnv(natural=True)
env.reset(seed=SEED)
agent = BlackjackAgent(env, env.action_space.n, epsilon_greedy_masked, seed=SEED)
rewards_split = fit_agent(agent, env)

## 6. Counting + splitting (Hi-Lo)

In [ ]:
env = BlackjackSplitEnv(natural=True, weights=COUNT_WEIGHTS_HI_LO)
env.reset(seed=SEED)
agent = BlackjackAgent(env, env.action_space.n, epsilon_greedy_masked, seed=SEED)
rewards_split_hi_lo = fit_agent(agent, env)

## Results

In [ ]:
df = pd.DataFrame({
    'threshold_baseline': baseline_rewards,
    'default': rewards_default,
    'double': rewards_double,
    'count_fractional': rewards_count,
    'count_hi_lo': rewards_count_hi_lo,
    'split_fractional': rewards_split,
    'split_hi_lo': rewards_split_hi_lo,
})
df.to_csv('../results/data/rewards.csv', index=False)

x = np.arange(len(df)) * STEP
plt.figure(figsize=(11, 6))
for col in df.columns:
    plt.plot(x, df[col], label=col)
plt.axhline(0, color='grey', lw=0.8, ls='--')
plt.xlabel('training games'); plt.ylabel('mean reward per game')
plt.title('Q-learning on Blackjack variants'); plt.legend()
plt.tight_layout(); plt.savefig('../results/figures/rewards_plot.png', dpi=150)
plt.show()